## ESTR2020 Project : Bayesian Matrix Factorization

In [ ]:
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

### Import and preprocess the dataset

In [ ]:
def data_preprocessing(file_path):

    ratings_df = pd.read_csv(file_path)
    
    unique_users = sorted(ratings_df['userId'].unique())
    unique_items = sorted(ratings_df['movieId'].unique())
    
    user_to_idx = {user: idx for idx, user in enumerate(unique_users)}
    item_to_idx = {item: idx for idx, item in enumerate(unique_items)}

    ratings_df['user_idx'] = ratings_df['userId'].map(user_to_idx)
    ratings_df['item_idx'] = ratings_df['movieId'].map(item_to_idx)
    
    ratings_df['norm_rating'] = (ratings_df['rating'] - 1) / 4.0  # Assuming ratings are from 1 to 5
    
    n_users = len(unique_users)
    n_items = len(unique_items)
    
    rating_matrix = sparse.lil_matrix((n_users, n_items))
    for _, row in ratings_df.iterrows():
        rating_matrix[row['user_idx'], row['item_idx']] = row['norm_rating']
    # The above steps are to make sure that user_idx and item_idx are unique and sequential starting from 0.
    

    # Create list of (user_id, item_id, rating) tuples
    ratings_data = list(zip(
        ratings_df['user_idx'].values,
        ratings_df['item_idx'].values,
        ratings_df['norm_rating'].values
    ))


    # Convert ratings_data into specific user and item indices
    ratings_by_user = {}
    ratings_by_item = {}
    
    for user_id, item_id, rating in ratings_data:

        if user_id not in ratings_by_user:
            ratings_by_user[user_id] = []
        ratings_by_user[user_id].append((item_id, rating))
        
        if item_id not in ratings_by_item:
            ratings_by_item[item_id] = []
        ratings_by_item[item_id].append((user_id, rating))
    
    
    # Create train/validation/test splits (80/10/10)
    train_data, test_data = train_test_split(ratings_data, test_size=0.2, random_state=42)
    val_data, test_data = train_test_split(test_data, test_size=0.5, random_state=42)
    
    return {
        'rating_matrix': rating_matrix,
        'ratings_data': ratings_data,
        'train_data': train_data,
        'val_data': val_data,
        'test_data': test_data,
        'n_users': n_users,
        'n_items': n_items,
        'ratings_by_user': ratings_by_user,
        'ratings_by_item': ratings_by_item
    }

## Class BayesianMatrixFactorization Structures

Implementation of Bayesian Matrix Factorization using Metropolis-Hastings algorithm for posterior sampling

In [ ]:
class BayesianMatrixFactorization:

    def __init__(self, n_users, n_items, n_factors=8, observation_noise=0.01, proposal_var_u=0.005, proposal_var_v=0.005, seed=42):

        self.n_users = n_users
        self.n_items = n_items
        self.n_factors = n_factors
        self.observation_noise = observation_noise
        self.proposal_var_u = proposal_var_u
        self.proposal_var_v = proposal_var_v

        np.random.seed(seed)
        
        self.U_samples = []
        self.V_samples = []


    # Method to be implemented
    def sigmoid(self, x): pass
    def init_matrices(self, rating_matrix): pass
    def log_prior(self, U, V): pass
    def log_individual_likelihood(self, user_factor, item_factor, rating): pass
    def log_likelihood(self, U, V, ratings_data): pass
    def log_unnormalized_posterior(self, U, V, ratings_data): pass
    def metropolis_step_user(self, user_id, U, V, ratings_data_by_user): pass
    def metropolis_step_item(self, item_id, U, V, ratings_data_by_item): pass
    def log_user_conditional(self, user_id, u_vec, V, ratings_data_by_user): pass
    def log_item_conditional(self, item_id, v_vec, U, ratings_data_by_item): pass
    def fit(self, rating_matrix, ratings_data, ratings_by_user, ratings_by_item, n_iterations=5000, burn_in=1000, thinning=5, verbose=True): pass
    def predict(self, user_id, item_id, rating): pass
    def MAP_estimation(self, user_id, item_id): pass
    def evaluate(self, test_data): pass

## Function implementations

### sigmoid : $ \sigma $

**Parameters:**
- x : array-like -> Input values (dot products of user and item latent factors)
  
**Returns:**
- array-like -> Sigmoid of input values, mapped to [0,1] range

This corresponds to the sigmoid function in equation (4) of the paper:

$$f(r_{ij}|u_i, v_j) = N(r_{ij}|\text{sigmoid}(u_i^T v_j), \sigma^2)$$

In [ ]:
def sigmoid(self, x):
    return 1 / (1 + np.exp(-x))

BayesianMatrixFactorization.sigmoid = sigmoid

### init_matrices -> to be update

**Parameters:**
- rating_matrix : sparse matrix -> Matrix containing observed ratings

**Returns:**
- tuple : Initial U and V matrices

Initializes matrices for the matrix factorization model R ≈ UVᵀ as described in section 1:
$$R \approx UV^T$$

In [ ]:
def init_matrices(self, rating_matrix):
    # Fill missing values with mean for SVD initialization
    dense_matrix = rating_matrix.toarray()
    mean_rating = np.nanmean(dense_matrix[dense_matrix != 0])
    dense_matrix[dense_matrix == 0] = mean_rating
    
    # Perform truncated SVD
    U, s, Vt = np.linalg.svd(dense_matrix, full_matrices=False)
    
    # Take only the first n_factors
    U = U[:, :self.n_factors]
    s_sqrt = np.sqrt(np.diag(s[:self.n_factors]))
    V = Vt.T[:, :self.n_factors]
    
    # Scale U and V by the singular values
    U = np.dot(U, s_sqrt)
    V = np.dot(V, s_sqrt)
    
    return U, V

BayesianMatrixFactorization.init_matrices = init_matrices

### log_prior : $ f(U) * f(V)$

**Parameters:**
- U : array-like -> User factors matrix
- V : array-like -> Item factors matrix

**Returns:**
- float : Log prior probability

Implements the standard normal priors based on equations (2) and (3) in the paper:
$$f(u_i) = \frac{1}{(2\pi)^{K/2}} \exp\left(-\frac{1}{2}u_i^T u_i\right)$$
$$f(v_j) = \frac{1}{(2\pi)^{K/2}} \exp\left(-\frac{1}{2}v_j^T v_j\right)$$

In [ ]:
def log_prior(self, U, V):

    log_prior_result = 0

    for idx in range(self.n_users):
        log_prior_result -= 0.5 * self.n_factors * np.log(2 * np.pi)
        log_prior_result -= 0.5 * np.dot(U[idx], U[idx])
    
    for idx in range(self.n_items):
        log_prior_result -= 0.5 * self.n_factors * np.log(2 * np.pi)
        log_prior_result -= 0.5 * np.dot(V[idx], V[idx])
    
    return log_prior_result

BayesianMatrixFactorization.log_prior = log_prior
    

### log_individual_likelihood : $ f(r_{ab}|u_{a}, v_{b}) $

**Parameters:**
- user_factor : array-like -> User factors vector
- item_factor : array-like -> Item factors vector
- rating : float -> Observed rating of specific user and item

**Returns:**
- float : Log likelihood of observed rating

Corresponds to equation (4) in the paper:
$$f(r_{ab}|u_{a}, v_{b}) = N(r_{ab}|\text{sigmoid}(u_a^T v_b), \sigma^2)$$

In [ ]:
def log_individual_likelihood(self, user_factor, item_factor, rating):

    pred = self.sigmoid(np.dot(user_factor, item_factor))

    return - 0.5 * np.log(2 * np.pi * self.observation_noise) - 0.5 * ((rating - pred) ** 2) / self.observation_noise

BayesianMatrixFactorization.log_individual_likelihood = log_individual_likelihood

### log_likelihood : $ f(\{r_{ij}\}|U, V) $

**Parameters:**
- U : array-like -> User factors matrix
- V : array-like -> Item factors matrix
- ratings_data : list of tuples -> List of (user_id, item_id, rating) tuples

**Returns:**
- float : Log likelihood of observed ratings

Corresponds to equation (6) in the paper:
$$f(\{r_{ij}\}|U, V) = \prod_{i=1}^{N} \prod_{j=1}^{M} [N(r_{ij}|\text{sigmoid}(u_i^T v_j), \sigma^2)]^{I_{ij}}$$

In [ ]:
def log_likelihood(self, U, V, ratings_data):
    log_likelihood = 0

    for user_id, item_id, rating in ratings_data:
        log_likelihood += self.log_individual_likelihood(U[user_id], V[item_id], rating)          
    
    return log_likelihood

BayesianMatrixFactorization.log_likelihood = log_likelihood

### log_unnormalized_posterior $ g(U, V) = f(\{r_{ij}\}|U, V) * f(U) * f(V)$

**Parameters:**
- U : array-like -> User factors matrix
- V : array-like -> Item factors matrix
- ratings_data : list of tuples -> List of (user_id, item_id, rating) tuples

**Returns:**
- float : Unnormalized log posterior probability

Implements the unnormalized posterior from equation (9):
$$f(U, V|\{r_{ij}\}) \propto \prod_{i=1}^{N} \prod_{j=1}^{M} [N(r_{ij}|\text{sigmoid}(u_i^T v_j), \sigma^2)]^{I_{ij}} \cdot \exp\left(-\frac{1}{2}\sum_{i=1}^{N} u_i^T u_i\right) \cdot \exp\left(-\frac{1}{2}\sum_{j=1}^{M} v_j^T v_j\right)$$

In [ ]:
def log_unnormalized_posterior(self, U, V, ratings_data):
    return self.log_prior(U, V) + self.log_likelihood(U, V, ratings_data)

BayesianMatrixFactorization.log_unnormalized_posterior = log_unnormalized_posterior

### log_user_conditional $ g(u_{i}) = f(u_{i}) * f(\{r_{ij}\}|u_{i}, V) $

This function will be used when perform the Metropolis-Hasting step

**Parameters:**
- user_id : int -> ID of the user
- u_vec : array-like -> User factor vector
- V : array-like -> Item factors matrix
- ratings_data_by_user : dict -> Dictionary mapping user_ids to their observed ratings

**Returns:**
- float : Log conditional probability

In [ ]:
def log_user_conditional(self, user_id, u_vec, V, ratings_data_by_user):
    # Prior component
    log_prior = -0.5 * np.dot(u_vec, u_vec)
    
    # Likelihood component
    log_likelihood = 0
    for item_id, rating in ratings_data_by_user[user_id]:
        pred = self.sigmoid(np.dot(u_vec, V[item_id]))
        log_likelihood += -0.5 * ((rating - pred) ** 2) / self.observation_noise
        log_likelihood += -0.5 * np.log(2 * np.pi * self.observation_noise)
    
    return log_prior + log_likelihood

BayesianMatrixFactorization.log_user_conditional = log_user_conditional

### log_item_conditional $ g(v_{j}) = f(v_{j}) * f(\{r_{ij}\}|U, v_{j})$

This function will be used when perform the Metropolis-Hasting step

**Parameters:**
- item_id : int -> ID of the item
- v_vec : array-like -> Item factor vector
- U : array-like -> User factors matrix
- ratings_data_by_item : dict -> Dictionary mapping item_ids to their observed ratings

**Returns:**
- float : Log conditional probability

In [ ]:
def log_item_conditional(self, item_id, v_vec, U, ratings_data_by_item):
    # Prior component
    log_prior = -0.5 * np.dot(v_vec, v_vec)
    
    # Likelihood component
    log_likelihood = 0
    for user_id, rating in ratings_data_by_item[item_id]:
        pred = self.sigmoid(np.dot(U[user_id], v_vec))
        log_likelihood += -0.5 * ((rating - pred) ** 2) / self.observation_noise
        log_likelihood += -0.5 * np.log(2 * np.pi * self.observation_noise)
    
    return log_prior + log_likelihood

BayesianMatrixFactorization.log_item_conditional = log_item_conditional

### metropolis_step_user

**Parameters:**
- user_id : int -> ID of the user to update
- U : array-like -> Current user factors matrix
- V : array-like -> Current item factors matrix
- ratings_data_by_user : dict -> Dictionary mapping user_ids to their observed ratings

**Returns:**
- array-like : Updated user factor vector
- bool : Whether the proposal was accepted

Implements the Metropolis-Hastings step for user factors as described in equations (30-31):
$$A(z^*, z) = \min\left(1, \frac{g(z^*)q(z|z^*)}{g(z)q(z^*|z)}\right)$$

In this case, we can reduce to:
$$ A(u_{i}^*, u_{i}) = \min\left(1, \frac{g(u_{i}^*)}{g(u_{i})}\right) $$

In [ ]:
def metropolis_step_user(self, user_id, U, V, ratings_data_by_user):

    u_current = U[user_id].copy()
    u_proposal = u_current + np.random.normal(0, np.sqrt(self.proposal_var_u), self.n_factors) # Adding Gaussian
    
    log_p_current = self.log_user_conditional(user_id, u_current, V, ratings_data_by_user)
    log_p_proposal = self.log_user_conditional(user_id, u_proposal, V, ratings_data_by_user)
    
    log_accept_prob = log_p_proposal - log_p_current
    
    # Accept or reject proposal
    if np.log(np.random.random()) < log_accept_prob:
        return u_proposal, True
    else:
        return u_current, False

BayesianMatrixFactorization.metropolis_step_user = metropolis_step_user

### metropolis_step_item

**Parameters:**
- item_id : int -> ID of the item to update
- U : array-like -> Current user factors matrix
- V : array-like -> Current item factors matrix
- ratings_data_by_item : dict -> Dictionary mapping item_ids to their observed ratings

**Returns:**
- array-like : Updated item factor vector
- bool : Whether the proposal was accepted

Implements the Metropolis-Hastings step for item factors as described in equations (30-31):
$$A(z^*, z) = \min\left(1, \frac{g(z^*)q(z|z^*)}{g(z)q(z^*|z)}\right)$$

In this case, we can reduce to:
$$ A(v_{j}^*, v_{j}) = \min\left(1, \frac{g(v_{j}^*)}{g(v_{j})}\right) $$

In [ ]:
def metropolis_step_item(self, item_id, U, V, ratings_data_by_item):

    v_current = V[item_id].copy()
    v_proposal = v_current + np.random.normal(0, np.sqrt(self.proposal_var_v), self.n_factors) # Adding Gaussian
    
    log_p_current = self.log_item_conditional(item_id, v_current, U, ratings_data_by_item)
    log_p_proposal = self.log_item_conditional(item_id, v_proposal, U, ratings_data_by_item)

    log_accept_prob = log_p_proposal - log_p_current
    
    # Accept or reject proposal
    if np.log(np.random.random()) < log_accept_prob:
        return v_proposal, True
    else:
        return v_current, False

BayesianMatrixFactorization.metropolis_step_item = metropolis_step_item

### fit

**Parameters:**
- rating_matrix : sparse matrix -> Matrix containing observed ratings
- ratings_data : list of tuples -> List of (user_id, item_id, rating) tuples
- n_iterations : int -> Number of MCMC iterations
- burn_in : int -> Number of initial samples to discard
- thinning : int -> Keep every 'thinning' samples
- show_progress : bool -> Whether to print progress information

**Returns:**
- self : The fitted model
- list : Log posterior values during sampling

**Statistical values:**
- `accepts_u`, `accepts_v` : acceptance rate -> used for fine-tuning parameter such as variance, acceptance rate mustn't be too high or too low
- `unnormalized_posteriors` -> The value should be increase during burn-in phase, after that it should be stable. It represents that our sample are converge to high-probability regions of the posterior

Implements the main MCMC sampling procedure as described in section 4.1:

In [ ]:
def fit(self, rating_matrix, ratings_data, ratings_by_user, ratings_by_item, n_iterations=5000, burn_in=1000, thinning=5, show_progress=True):

    U, V = self.init_matrices(rating_matrix)
    
    accepts_u = 0
    accepts_v = 0
    total_u = 0
    total_v = 0
    
    log_unnormalized_posteriors = []
    
    # clear previous samples
    self.U_samples = []
    self.V_samples = []
    
    
    for iteration in range(n_iterations):

        # Update user factors
        for user_id in range(self.n_users):
            U[user_id], accepted = self.metropolis_step_user(user_id, U, V, ratings_by_user)
            accepts_u += accepted
            total_u += 1
        
        # Update item factors
        for item_id in range(self.n_items):
            V[item_id], accepted = self.metropolis_step_item(item_id, U, V, ratings_by_item)
            accepts_v += accepted
            total_v += 1
        

        # Tracking log posterior
        log_unnormalized_posteriors.append(self.log_unnormalized_posterior(U, V, ratings_data))
        

        # Store samples after burn-in, using thinning
        if iteration >= burn_in and (iteration - burn_in) % thinning == 0:
            self.U_samples.append(U.copy())
            self.V_samples.append(V.copy())

        
        if show_progress and (iteration + 1) % 100 == 0:
            print(f"Iteration {iteration + 1}/{n_iterations}, Log posterior: {log_unnormalized_posteriors[-1]:.4f}")
            print(f"Acceptance rates - Users: {accepts_u / total_u:.4f}, Items: {accepts_v / total_v:.4f}")
    
    if show_progress:
        print(f"Sampling completed. Final acceptance rates - Users: {accepts_u / total_u:.4f}, Items: {accepts_v / total_v:.4f}")
        print(f"Collected {len(self.U_samples)} samples after burn-in and thinning")
        

    return self, log_unnormalized_posteriors

BayesianMatrixFactorization.fit = fit

### predict

**Parameters:**
- user_id : int -> ID of the user
- item_id : int -> ID of the item
- rating : float -> Rating to be predicted

**Returns:**
- float : The probability of $ r_{ab} = rating$

Implements prediction using collected posterior samples as in equation (39):
$$f(r_{ab}|\{r_{ij}\}) \approx \frac{1}{S} \sum_{s=1}^{S} N(r_{ab}|\text{sigmoid}((u_a^{(s)})^T v_b^{(s)}), \sigma^2)$$

In [ ]:
def predict(self, user_id, item_id, rating):
    predictions = []

    for U_sample, V_sample in zip(self.U_samples, self.V_samples):
        predictions.append(self.individual_likelihood(U_sample[user_id], V_sample[item_id], rating))
    
    return np.mean(predictions)

BayesianMatrixFactorization.predict = predict

### MAP_estimation

**Parameters:**
- user_id : int -> ID of the user
- item_id : int -> ID of the item

**Returns:**
- int -> Predicted rating from [1, 2, 3, 4, 5]

Implements Maximum A Posteriori (MAP) as in equation (12):
$$r^* = \arg\max_r f(r_{ab} \mid \{r_{ij}\})$$

In [ ]:
def MAP_estimation(self, user_id, item_id):

    predicted_prob = []
    for i in range(5):
        predicted_prob.append(self.predict(user_id, item_id, i / 4.0))

    return np.argmax(predicted_prob) + 1

BayesianMatrixFactorization.MAP_estimation = MAP_estimation

### evaluate

**Parameters:**
- test_data : list of tuples -> List of (user_id, item_id, rating) tuples

**Returns:**
- int : The number of correct predictions
- float : Root Mean Squared Error (RMSE)

In [ ]:
def evaluate(self, test_data):
    
    error = 0.0
    correct = 0
    for user_id, item_id, rating in test_data:
        pred = self.MAP_estimation(user_id, item_id)
        error += (rating - pred) ** 2
    
    return correct, np.sqrt(error / len(test_data))

BayesianMatrixFactorization.evaluate = evaluate

## Program simulation

In [ ]:
data = data_preprocessing('ml-latest-small/ratings.csv')

bmf = BayesianMatrixFactorization(
    n_users=data['n_users'],
    n_items=data['n_items'],
    n_factors=8,
    observation_noise=0.01,
    proposal_var_u=0.005,
    proposal_var_v=0.005
)

bmf, log_posteriors = bmf.fit(
    data['rating_matrix'],
    data['train_data'],
    data['ratings_by_user'],
    data['ratings_by_item'],
    n_iterations=5000,
    burn_in=1000,
    thinning=5
)

val_correct, val_rmse = bmf.evaluate(data['val_data'])
test_correct, test_rmse = bmf.evaluate(data['test_data'])

print(f"Validation Correct Prediction: {val_correct:.4f}, Validation RMSE: {val_rmse:.4f}")
print(f"Train Correct Prediction: {test_correct:.4f}, Test RMSE: {test_rmse:.4f}")

## Result Visualization

In [ ]:
# Plot trace of log posterior
plt.figure(figsize=(10, 6))
plt.plot(log_posteriors)
plt.title('Trace of Log Posterior')
plt.xlabel('Iteration')
plt.ylabel('Log Posterior')
plt.grid(True)
plt.show()